# AutoGluon: AutoML for Tabular, Text, and Vision

## What Is AutoGluon?

Imagine you're a new chef who needs to serve the best possible dish.  
A senior chef (AutoGluon) takes your ingredients (data), tries 20 different recipes (models) automatically,
and combines the best ones into a super-dish (ensemble) — all while you wait.

**AutoGluon** is Amazon's open-source AutoML library.  
Give it a dataset and a target column — it trains dozens of models, tunes them, and stacks them into an ensemble,
often beating hand-tuned models with just **3 lines of code**.

Key capabilities:
- **Tabular**: CSV/DataFrame → best model automatically
- **Text**: NLP tasks (classification, regression on text)
- **Image**: computer vision classification
- **Multimodal**: combine tabular + text + images
- **Stack ensembling**: layer models on top of each other for max accuracy

## Resources

- **Docs**: [https://auto.gluon.ai/](https://auto.gluon.ai/)
- **GitHub**: [https://github.com/autogluon/autogluon](https://github.com/autogluon/autogluon)
- **YouTube**: [https://www.youtube.com/watch?v=gG1O7tiBEe8](https://www.youtube.com/watch?v=gG1O7tiBEe8)
- **Paper**: [AutoGluon-Tabular: Robust and Accurate AutoML](https://arxiv.org/abs/2003.06505)

## Installation

```bash
# Tabular only (fastest install)
pip install autogluon.tabular

# All modalities
pip install autogluon

# Note: large install (~2GB). For CPU only:
pip install autogluon.tabular[all]
```

In [ ]:
import numpy as np
import time

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    from autogluon.tabular import TabularPredictor
    AG_AVAILABLE = True
    import autogluon
    print(f"AutoGluon version: {autogluon.__version__}")
except ImportError:
    AG_AVAILABLE = False
    print("AutoGluon not installed — simulated output shown.")
    print("Install: pip install autogluon.tabular")

# Create a synthetic classification dataset
np.random.seed(42)
N_TRAIN, N_TEST = 1000, 200

def make_dataset(n, seed=42):
    rng = np.random.RandomState(seed)
    age      = rng.randint(18, 75, n)
    income   = rng.exponential(50000, n).round(0)
    tenure   = rng.randint(0, 120, n)       # months with company
    usage    = rng.poisson(30, n)            # product usage per month
    category = rng.choice(['A', 'B', 'C'], n)
    # Churn: high income + low usage = more likely to churn
    churn_prob = 1 / (1 + np.exp(-(0.3*(income/50000 - 1) - 0.2*(usage - 30)/30 + rng.normal(0, 0.5, n))))
    churn    = (churn_prob > 0.5).astype(int)
    if PANDAS_AVAILABLE:
        return pd.DataFrame({'age': age, 'income': income, 'tenure': tenure,
                             'usage': usage, 'category': category, 'churn': churn})
    return None

if PANDAS_AVAILABLE:
    train_data = make_dataset(N_TRAIN, seed=42)
    test_data  = make_dataset(N_TEST,  seed=99)
    print(f"\nTrain: {N_TRAIN} rows, Test: {N_TEST} rows")
    print(f"Churn rate: {train_data['churn'].mean():.1%}")
    print(train_data.head())

## Core Concept 1: TabularPredictor — AutoML in 3 Lines

AutoGluon's `TabularPredictor` is the simplest way to get a strong ML model.  
It automatically: detects the problem type, imputes missing values, encodes categoricals, trains dozens of models, and stacks them.

In [ ]:
import os, tempfile

if AG_AVAILABLE and PANDAS_AVAILABLE:
    save_dir = os.path.join(tempfile.mkdtemp(), 'autogluon_churn')

    t0 = time.time()
    # 3 lines of AutoML!
    predictor = TabularPredictor(
        label='churn',           # target column
        path=save_dir,           # where to save models
        eval_metric='roc_auc',   # optimize for AUC
        verbosity=1,             # 0=silent, 2=verbose
    ).fit(
        train_data,
        time_limit=60,           # train for max 60 seconds
        presets='medium_quality' # 'best_quality' takes longer
    )
    elapsed = time.time() - t0

    print(f"\nTraining complete in {elapsed:.0f}s")

    # Evaluate on test set
    y_test = test_data['churn']
    X_test = test_data.drop('churn', axis=1)

    perf = predictor.evaluate(test_data)
    print(f"\nTest performance: {perf}")

    # Make predictions
    predictions = predictor.predict(X_test)
    probabilities = predictor.predict_proba(X_test)
    print(f"\nSample predictions:\n{predictions[:5].values}")
    print(f"\nSample probabilities:\n{probabilities[:5]}")

else:
    print("AutoGluon TabularPredictor (simulated):")
    print()
    print("  predictor = TabularPredictor(")
    print("      label='churn',")
    print("      eval_metric='roc_auc',")
    print("  ).fit(")
    print("      train_data,")
    print("      time_limit=60,")
    print("      presets='medium_quality'")
    print("  )")
    print()
    print("  AutoGluon is training the following models:")
    print("    LightGBM, XGBoost, CatBoost, RandomForest, ExtraTrees,")
    print("    KNN, NeuralNetwork, WeightedEnsemble_L2")
    print()
    print("  Test performance:")
    print("    roc_auc: 0.892")
    print("    accuracy: 0.825")

## Core Concept 2: Leaderboard — See All Trained Models

AutoGluon trains many models and ranks them. The **leaderboard** shows all models and their scores.

In [ ]:
if AG_AVAILABLE and PANDAS_AVAILABLE:
    print("Model Leaderboard (all trained models):")
    leaderboard = predictor.leaderboard(test_data, silent=True)
    print(leaderboard[['model', 'score_test', 'score_val', 'fit_time', 'pred_time_test']].to_string())
    print()

    # Best model info
    print(f"Best model: {predictor.model_best}")
    print(f"Feature importance:")
    fi = predictor.feature_importance(test_data)
    print(fi)

else:
    print("Leaderboard (simulated):")
    print()
    print("  predictor.leaderboard(test_data)")
    print()
    print("  model                  score_test  score_val  fit_time  pred_time")
    print("  WeightedEnsemble_L2       0.892      0.887      52.3      0.08")
    print("  LightGBM                  0.885      0.882      12.1      0.02")
    print("  XGBoost                   0.881      0.878       8.7      0.03")
    print("  CatBoost                  0.879      0.876      15.2      0.04")
    print("  RandomForest_gini         0.868      0.862       6.3      0.05")
    print("  ExtraTreesGini            0.861      0.857       4.2      0.04")
    print("  NeuralNetTorch            0.854      0.849      18.9      0.06")
    print("  KNeighborsUnif            0.812      0.809       0.2      0.12")
    print()
    print("  Feature Importance (permutation-based):")
    print("    usage:    0.187  ← most important")
    print("    income:   0.143")
    print("    tenure:   0.098")
    print("    age:      0.067")
    print("    category: 0.034")

## Core Concept 3: Quality Presets

AutoGluon has presets that trade off training time vs. accuracy:

| Preset | Time | Accuracy | When to use |
|--------|------|----------|-------------|
| `medium_quality` | Fast (~1 min) | Good | Quick experiments |
| `good_quality` | Moderate (~5 min) | Better | Default choice |
| `high_quality` | Long (~30 min) | High | Before deployment |
| `best_quality` | Very long (~hours) | Best | Competitions, prod |
| `optimize_for_deployment` | Fast | Same | Smaller, faster models |

In [ ]:
print("AutoGluon presets and what they do:")
print()

presets_info = {
    'medium_quality': {
        'models': ['LightGBM', 'XGBoost', 'RF', 'XT', 'KNN', 'NNTorch'],
        'stack_layers': 1,
        'hpo': False,
        'typical_time': '1-3 min',
        'vs_baseline': '+5-15%'
    },
    'good_quality': {
        'models': ['LightGBM×5', 'XGBoost×5', 'CatBoost', 'RF', 'XT', 'KNN', 'NNTorch×3'],
        'stack_layers': 2,
        'hpo': False,
        'typical_time': '5-15 min',
        'vs_baseline': '+8-20%'
    },
    'best_quality': {
        'models': 'All + HPO for each',
        'stack_layers': 3,
        'hpo': True,
        'typical_time': '1-12 hours',
        'vs_baseline': '+15-30%'
    },
}

for preset, info in presets_info.items():
    print(f"  Preset: '{preset}'")
    print(f"    Models:       {info['models']}")
    print(f"    Stack layers: {info['stack_layers']}")
    print(f"    HPO:          {info['hpo']}")
    print(f"    Typical time: {info['typical_time']}")
    print(f"    vs manual:    {info['vs_baseline']} AUC gain")
    print()

print("Usage:")
print("  predictor = TabularPredictor(label='y').fit(")
print("      train_data,")
print("      presets='best_quality',   # or any of the above")
print("      time_limit=3600,          # 1 hour max")
print("  )")

## Core Concept 4: Stack Ensembling — How AutoGluon Wins

AutoGluon's secret weapon is **multi-layer stack ensembling**:

```
Layer 1 (Base models):
  LightGBM → pred_LGB
  XGBoost  → pred_XGB
  RF       → pred_RF

Layer 2 (Stack models — trained on Layer 1 predictions + original features):
  LightGBM(pred_LGB, pred_XGB, pred_RF, original_features) → pred_stacked

Layer 3 (Weighted ensemble of all):
  WeightedEnsemble(all_predictions) → final_prediction
```

Each layer corrects the errors of the previous one.

In [ ]:
print("Stack Ensembling illustration:")
print()
print("  Training data: 1000 rows × 5 features")
print()
print("  === Layer 1: Base models (5-fold cross-val OOF predictions) ===")
print("  LightGBM      → val_AUC: 0.882  (out-of-fold predictions for Layer 2)")
print("  XGBoost       → val_AUC: 0.878")
print("  CatBoost      → val_AUC: 0.876")
print("  RandomForest  → val_AUC: 0.862")
print("  NeuralNetwork → val_AUC: 0.849")
print()
print("  === Layer 2: Stack models (use L1 OOF predictions as extra features) ===")
print("  Features = [age, income, tenure, usage, cat, LGB_pred, XGB_pred, ...]")
print("  LightGBM_L2   → val_AUC: 0.891  (better! uses L1 errors as signals)")
print("  XGBoost_L2    → val_AUC: 0.888")
print()
print("  === Layer 3: Weighted Ensemble (optimal linear blend) ===")
print("  0.4 × LightGBM_L2 + 0.3 × XGBoost_L2 + 0.2 × LightGBM + ...")
print("  WeightedEnsemble_L2 → val_AUC: 0.896  ← best!")
print()
print("  Key insight: 'out-of-fold' predictions avoid data leakage.")
print("  Each base model only predicts on rows it never trained on.")

if AG_AVAILABLE and PANDAS_AVAILABLE:
    print()
    print("Actual model info from trained predictor:")
    info = predictor.info()
    for model_name, model_info in list(info['model_info'].items())[:3]:
        print(f"  {model_name}: {model_info.get('val_score', 'N/A')}")

## Core Concept 5: Loading, Predicting, and Deployment

In [ ]:
if AG_AVAILABLE and PANDAS_AVAILABLE:
    # Reload predictor from disk (works across sessions)
    predictor_reloaded = TabularPredictor.load(save_dir)
    print("Predictor reloaded from disk!")

    # Fast predictions
    preds = predictor_reloaded.predict(X_test)
    print(f"Predictions: {preds[:5].values}")

    # Predict using only fast models (for low-latency serving)
    preds_fast = predictor_reloaded.predict(
        X_test,
        model='LightGBM'  # use a specific model, skipping ensemble overhead
    )
    print(f"Fast (single model) predictions: {preds_fast[:5].values}")

    # List available models
    print(f"\nAvailable models: {predictor_reloaded.model_names()}")

    # Clean up
    import shutil
    shutil.rmtree(save_dir, ignore_errors=True)

else:
    print("Loading and deployment (simulated):")
    print()
    print("  # Save happens automatically during .fit()")
    print("  # Reload in a new session:")
    print("  predictor = TabularPredictor.load('path/to/save_dir')")
    print()
    print("  # Predict")
    print("  predictor.predict(test_df)")
    print("  predictor.predict_proba(test_df)  # probability scores")
    print()
    print("  # For low-latency serving: use single fast model")
    print("  predictor.predict(X_test, model='LightGBM')  # skip ensemble")
    print()
    print("  # Optimize for deployment (refit, prune slow models)")
    print("  predictor.compile_models()")
    print("  predictor.get_model_best()  # best single model for deployment")

## Core Concept 6: Regression and Multiclass

AutoGluon automatically detects: binary classification, multiclass classification, or regression.

In [ ]:
if PANDAS_AVAILABLE:
    # Regression example
    np.random.seed(42)
    reg_train = pd.DataFrame({
        'sqft':    np.random.randint(500, 5000, 500),
        'beds':    np.random.randint(1, 6, 500),
        'baths':   np.random.randint(1, 4, 500),
        'age':     np.random.randint(0, 100, 500),
        'garage':  np.random.choice([0, 1, 2], 500),
        'price':   np.random.normal(300000, 80000, 500).round(-3)
    })
    reg_test = reg_train.sample(100, random_state=99)
    print("Regression dataset (house prices):")
    print(reg_train.head(3))
    print(f"Price range: ${reg_train['price'].min():,.0f} - ${reg_train['price'].max():,.0f}")
    print()

if AG_AVAILABLE and PANDAS_AVAILABLE:
    reg_dir = os.path.join(tempfile.mkdtemp(), 'ag_regression')
    reg_predictor = TabularPredictor(
        label='price',
        eval_metric='rmse',       # AutoGluon detects regression automatically
        path=reg_dir,
        verbosity=0,
    ).fit(reg_train, time_limit=30)

    reg_perf = reg_predictor.evaluate(reg_test)
    print(f"Regression performance: {reg_perf}")
    print(f"Problem type: {reg_predictor.problem_type}")
    shutil.rmtree(reg_dir, ignore_errors=True)

else:
    print("Regression (simulated):")
    print()
    print("  reg_predictor = TabularPredictor(")
    print("      label='price',")
    print("      eval_metric='rmse',  # AutoGluon detects regression")
    print("  ).fit(reg_train, time_limit=30)")
    print()
    print("  Performance:")
    print("    rmse: 52,340  (same as sklearn RMSE, lower is better)")
    print("    r2:   0.574")
    print()
    print("  AutoGluon auto-detects problem type:")
    print("    2 unique values in label  → binary classification")
    print("    3-10 unique values        → multiclass classification")
    print("    many float values         → regression")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Not setting `time_limit` | Trains for hours | Always set `time_limit` in seconds |
| Using `best_quality` for exploration | Takes hours per experiment | Use `medium_quality` for experiments, `best_quality` before final deployment |
| Leaking labels in features | Overly optimistic leaderboard | Check your features — never include the target or derived features |
| Not saving to disk | Model lost after session | `path=` parameter saves automatically |
| Large memory use | OOM on laptop | Reduce `num_bag_folds` or use `presets='medium_quality'` |
| Using ensemble in low-latency serving | Slow inference | Use `predictor.predict(X, model='LightGBM')` for single-model speed |

## Mini Project: Churn Prediction vs Manual Pipeline

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

if PANDAS_AVAILABLE:
    # Manual ML pipeline
    X_train = train_data.drop('churn', axis=1).copy()
    y_train = train_data['churn']
    X_t     = test_data.drop('churn', axis=1).copy()
    y_t     = test_data['churn']

    # Encode categorical
    le = LabelEncoder()
    X_train['category'] = le.fit_transform(X_train['category'])
    X_t['category']     = le.transform(X_t['category'])

    print("=" * 60)
    print("AUTOGLUON vs MANUAL PIPELINE — CHURN PREDICTION")
    print("=" * 60)
    print()

    # Manual models
    manual_models = {
        'RandomForest (default)': RandomForestClassifier(random_state=42),
        'GradientBoosting (default)': GradientBoostingClassifier(random_state=42),
    }

    for name, model in manual_models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        auc = roc_auc_score(y_t, model.predict_proba(X_t)[:, 1])
        print(f"  Manual {name}:")
        print(f"    AUC: {auc:.4f}  ({time.time()-t0:.1f}s)")

    print()
    if AG_AVAILABLE:
        ag_dir = os.path.join(tempfile.mkdtemp(), 'ag_compare')
        t0 = time.time()
        ag = TabularPredictor(label='churn', eval_metric='roc_auc',
                              path=ag_dir, verbosity=0).fit(
            train_data, time_limit=60, presets='medium_quality'
        )
        ag_perf = ag.evaluate(test_data)['roc_auc']
        print(f"  AutoGluon (medium, 60s limit):")
        print(f"    AUC: {ag_perf:.4f}  ({time.time()-t0:.1f}s)")
        shutil.rmtree(ag_dir, ignore_errors=True)
    else:
        print("  AutoGluon (simulated result):")
        print(f"    AUC: 0.8920  (52s)")

    print()
    print("  Conclusion: AutoGluon outperforms individual models with")
    print("  less manual effort — no hyperparameter tuning needed!")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "What is AutoML and what are its trade-offs?",
     "a": """AutoML (Automated Machine Learning) automates the ML pipeline:
  model selection, hyperparameter tuning, feature engineering, ensembling.

Benefits:
  - Faster iteration: hours → minutes for a baseline
  - Democratization: non-ML-experts can get strong models
  - Often beats hand-tuned models (especially with ensembling)

Trade-offs:
  - Black box: hard to understand why it made certain choices
  - Resource intensive: trains many models → high compute cost
  - Less control: can't inject domain knowledge into the pipeline
  - Overfitting risk: with many models, leakage is possible
  - Latency: ensemble models are slow for real-time serving

When NOT to use AutoML:
  - Interpretability required (regulatory, medical)
  - Latency < 10ms (ensemble too slow)
  - Domain knowledge critical (feature engineering matters more)
  - Very small datasets (< 100 rows) — models overfit easily"""},

    {"q": "How does AutoGluon's stack ensembling avoid data leakage?",
     "a": """Stack ensembling trains Layer 2 models on Layer 1's predictions.
Naive approach would leak: train LightGBM, get predictions on train set
(which LightGBM saw during training) → Layer 2 sees 'perfect' features → overfits.

AutoGluon uses out-of-fold (OOF) cross-validation:
  1. Split train data into 5 folds
  2. For each fold: train LightGBM on 4 folds, predict on 1 (held-out)
  3. Collect predictions for all 5 held-out folds → OOF predictions
  4. Each OOF prediction was made on data the model NEVER saw
  5. Layer 2 trains on OOF predictions → no leakage

Result: Layer 2 sees realistic predictions (not memorized train predictions),
so it genuinely learns to correct Layer 1's errors.

This is why stacking > simple averaging: each layer fixes the previous."""},

    {"q": "AutoGluon vs H2O AutoML vs sklearn GridSearchCV — when to use each?",
     "a": """sklearn GridSearchCV:
  - When: you know which model to use, just need tuning
  - Pro: simple, familiar, integrates with sklearn
  - Con: exhaustive (slow), single model, no ensembling
  - Use: fine-tune a chosen model, small hyperparameter spaces

H2O AutoML:
  - When: enterprise environment, Java/distributed cluster available
  - Pro: scalable, great for large datasets, MOJO export for serving
  - Con: Java dependency, setup complexity, paid cloud option
  - Use: large-scale production ML, Spark/cluster integration

AutoGluon:
  - When: best accuracy with minimal effort (Kaggle, rapid prototyping)
  - Pro: state-of-the-art ensembling, easy API, multimodal support
  - Con: heavy install, slow ensembles at inference time
  - Use: getting the best baseline, Kaggle competitions, PoC demos

Rule:
  Best accuracy, don't care about speed → AutoGluon
  Large-scale distributed → H2O
  Simple tuning of a known model → sklearn GridSearch/RandomSearch"""},

    {"q": "How would you deploy an AutoGluon model for production serving?",
     "a": """Production deployment has two challenges: latency and portability.

Option 1: Single model (fast, simple)
  predictor.predict(X, model='LightGBM')  # skip ensemble
  → latency: ~5ms per batch
  → Wrap in FastAPI: predictor.predict(pd.DataFrame([request.features]))

Option 2: Compile ensemble (moderate latency)
  predictor.compile_models()  # optimize model code
  → latency: ~50ms for full ensemble

Option 3: Export to ONNX (if supported)
  Not all AutoGluon models support ONNX yet.

Practical deployment:
  1. predictor.save()  # save all models
  2. Add to Docker: COPY save_dir/ /app/models/
  3. In FastAPI:
     predictor = TabularPredictor.load('/app/models/')
     @app.post('/predict')
     def predict(features: dict):
         df = pd.DataFrame([features])
         return predictor.predict(df, model='LightGBM').tolist()

Best practice: use single fast model for serving; ensemble only for batch."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Step | AutoGluon API |
|------|---------------|
| Create predictor | `TabularPredictor(label='col', eval_metric='roc_auc')` |
| Train | `.fit(train_df, time_limit=60, presets='medium_quality')` |
| Predict labels | `predictor.predict(test_df)` |
| Predict probabilities | `predictor.predict_proba(test_df)` |
| Evaluate | `predictor.evaluate(test_df)` |
| Leaderboard | `predictor.leaderboard(test_df)` |
| Feature importance | `predictor.feature_importance(test_df)` |
| Best model name | `predictor.model_best` |
| Save | Automatic (to `path=` directory) |
| Load | `TabularPredictor.load('path/')` |
| Single-model predict | `predictor.predict(X, model='LightGBM')` |

### Next Steps
1. **AutoGluon tabular quickstart**: [https://auto.gluon.ai/stable/tutorials/tabular/tabular-quick-start.html](https://auto.gluon.ai/stable/tutorials/tabular/tabular-quick-start.html)
2. **AutoGluon multimodal**: [https://auto.gluon.ai/stable/tutorials/multimodal/index.html](https://auto.gluon.ai/stable/tutorials/multimodal/index.html)
3. **Next**: H2O AutoML — enterprise-grade distributed AutoML